### In-context Learning
- 모델이 가중치를 바꾸지 않고 프롬프트에 예시를 추가해서 새로운 태스크를 수행하는 방식
- 새로운 태스크가 나타나면
    - 몇 줄의 예시 작성
    - 즉시 사용

### 기존 방식 - Fine turning
- 새로운 태스크가 나타나면
    - 데이터 수집 및 레이블링
    - 모델 재학습(며칠~주)
    - 배포

### Zero-shot vs One-shot VS Few-shot
- Zero-shot : 예시가 없음
- "다음 영화 리뷰의 감성을 분석하세요.
- 리뷰 : '이 영화는 정말 멋있었어!'
- 감성:"

- --> "긍정"

- 특징
    - 인스트럭션만으로 수행
    - 작은모델은 거의 작동안함
    - 큰 모델은 기본 테스크는 수행 가능

### One-shot : 1개 예시
```
프롬프트:
"다음 영화 리뷰의 감성을 분석하세요.

리뷰:'영화가 정말 재미있었어요!'
감성: 긍정

리뷰: '이 영화는 멋있었어!'
감성"
```
- 출력 : 긍정

- 특징:
    - 1개 예시제공으로 패턴 이해
    - 성능 급상승
    - 큰 모델에서 특히 효과적

### Few-shot(여러 예시)
- 특징
    - 다양한 예시로 강화
    - 최고의 성능
    - 일반적으로는 3 ~5개 최적

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [2]:
train_data = [
    ("영화가 정말 재미있다", "긍정"),
    ("최악의 영화였다", "부정"),
    ("배우 연기가 훌륭했다", "긍정"),
    ("스토리가 너무 지루했다", "부정"),
    ("정말 감동적이었다", "긍정"),
    ("시간 낭비였다", "부정"),
]

test_reviews = [
    "이 영화는 정말 멋있었어!",
    "완전히 실망했다",
    "그냥 볼만했다"
]

In [3]:
# zero-shot prompt
zero_shot_prompt = '''
다음 영화 리뷰의 감성을 분석하세요

리뷰:'이 영화는 정말 멋있었어!'
감성:
'''
one_shot_prompt = '''
다음 영화 리뷰의 감성을 분석하세요.

리뷰: "영화가 정말 재미있었어요!"
감성: 긍정

리뷰: "이 영화는 정말 멋있었어!"
감성:
'''
few_shot_prompt = '''
다음 영화 리뷰의 감성을 분석하세요.

리뷰: "영화가 정말 재미있었어요!"
감성: 긍정

리뷰: "완전히 실망했어요."
감성: 부정

리뷰: "그런대로 괜찮네요."
감성: 중립

리뷰: "이 영화는 정말 멋있었어!"
감성:
'''

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

In [5]:
%pip install accelerate

In [6]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
print(model)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [7]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
def generate_sentiment(prompt, max_new_tokens=20):
    message = [
        {
            "role" : "system",
            'content' : '너는 영화 리뷰 감성을 긍정, 부정, 중립중 하나로만 분류하는 분류기입니다.'
        },
        {
            'role':'user',
            'content':prompt
        }
    ]
    text = tokenizer.apply_chat_template(
        message,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text,return_tensors='pt').to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated_ids = output_ids[0][ inputs['input_ids'].shape[-1] : ]
    return tokenizer.decode(generated_ids,skip_special_tokens=True).strip()

In [9]:
results = {
    'Zero-shot' : generate_sentiment(zero_shot_prompt),
    'One-shot' : generate_sentiment(one_shot_prompt),
    'Few-shot' : generate_sentiment(few_shot_prompt),
}
for key,value in results.items():
    print(key,value)

Zero-shot 감성: 부정

이 리뷰에서 "이 영화는 정말 멋있었어
One-shot 감성: 부정
Few-shot 감성: 긍정
